# Step 1 — Similarity retention after attachment

This experiment asks whether similarity relationships among the 369 building blocks are retained after each building block is attached to one of ten constant partners. The same records and molecular pairs are used for every descriptor.

The four descriptors are: **ECFP-O** (the `[Hg]` marker is replaced by O), **ECFP-Hg** (the marker is retained as a control), **COAF**, and the **directed linear fingerprint**. Spearman correlation is the primary global measure; top-5 and top-10 neighbor preservation measure local retrieval. Pearson correlation is shown as a secondary measure.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import coaf
from validation.similarity_retention import (
    BenchmarkConfig,
    load_similarity_retention_dataset,
    run_similarity_retention_benchmark,
)

PROJECT_ROOT = Path.cwd()
DATA_FILE = PROJECT_ROOT / 'data' / 'SMILES_1aa.csv'
RESULT_DIR = PROJECT_ROOT / 'results' / 'step1'
FIGURE_DIR = RESULT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('COAF implementation:', coaf.__file__)

## Load and validate the dataset

The dataset loader checks the expected column structure, RDKit parsing, connectivity, exactly one Hg marker per structure, and unique building-block SMILES. It adds analysis-only building-block IDs while leaving the raw CSV unchanged.

In [ ]:
dataset = load_similarity_retention_dataset(DATA_FILE)

print(f'Building blocks: {dataset.n_molecules}')
print(f'Attachment partners: {len(dataset.product_columns)}')
print('Product columns:', ', '.join(dataset.product_columns))
display(dataset.frame[['bb_id', 'source_row', 'SMILES_BB']].head())

The displayed rows are the molecule manifest: `bb_id` is the persistent analysis identifier, `source_row` points back to the original CSV line, and `SMILES_BB` is the unmodified rooted building-block structure.

## Run the benchmark

The primary configuration uses 1,024-bit fingerprints, ECFP radius 3, COAF radius 3, and a POAF maximum path length of six edges (POAF6). Results and similarity matrices are saved under `results/step1`.

In [ ]:
config = BenchmarkConfig(
    n_bits=1024,
    ecfp_radius=3,
    coaf_radius=3,
    linear_max_path_length=6,
    neighbor_k=(5, 10),
)

results = run_similarity_retention_benchmark(
    dataset,
    config=config,
    output_dir=RESULT_DIR,
)

## Partner-specific results

Each row below corresponds to one descriptor and one constant attachment partner. Spearman and Pearson compare all unique building-block pairs with the corresponding product pairs. Neighbor preservation is the mean fraction of the original top-k neighbors retained after attachment.

In [ ]:
display(results.per_partner_metrics.round(4))

## Plotting functions

These visualization functions remain in the notebook because they control presentation rather than the scientific calculations. The saved similarity matrices are loaded directly, so the correlation plots represent the same values used in the result tables.

In [ ]:
DESCRIPTOR_LABELS = {
    'ECFP_O': 'ECFP-O',
    'ECFP_HG': 'ECFP-Hg',
    'COAF': 'COAF',
    'DIRECTED_LINEAR': 'Directed linear',
}
DESCRIPTOR_COLORS = {
    'ECFP_O': '#4C78A8',
    'ECFP_HG': '#72B7B2',
    'COAF': '#E45756',
    'DIRECTED_LINEAR': '#F2CF5B',
}

def _pairwise_values(matrix):
    return matrix[np.triu_indices_from(matrix, k=1)]

def save_figure(fig, filename, png_dpi=600):
    """Save a quantitative figure as editable SVG and high-resolution PNG."""
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    stem = Path(filename).stem
    svg_path = FIGURE_DIR / f'{stem}.svg'
    png_path = FIGURE_DIR / f'{stem}.png'
    fig.savefig(svg_path, format='svg', bbox_inches='tight')
    fig.savefig(png_path, format='png', dpi=png_dpi, bbox_inches='tight')
    print(f'Saved figure: {svg_path}')
    print(f'Saved figure: {png_path}')
    return {'svg': svg_path, 'png': png_path}

def plot_correlation_panels(result_dir, product_column, descriptors=None):
    """Plot BB versus product similarities for one attachment partner."""
    descriptors = descriptors or list(DESCRIPTOR_LABELS)
    fig, axes = plt.subplots(1, len(descriptors), figsize=(4.2 * len(descriptors), 3.8), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)
    matrix_dir = Path(result_dir) / 'similarity_matrices'

    for ax, descriptor in zip(axes, descriptors):
        bb = np.load(matrix_dir / f'{descriptor}__SMILES_BB.npy')
        product = np.load(matrix_dir / f'{descriptor}__{product_column}.npy')
        x = _pairwise_values(bb)
        y = _pairwise_values(product)
        rho = spearmanr(x, y).statistic

        plot = ax.hexbin(x, y, gridsize=45, mincnt=1, cmap='viridis')
        ax.plot([0, 1], [0, 1], '--', color='0.55', linewidth=1)
        ax.set_title(f'{DESCRIPTOR_LABELS[descriptor]}\nSpearman ρ = {rho:.3f}')
        ax.set_xlabel('Building-block similarity')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        fig.colorbar(plot, ax=ax, label='Pair count')

    axes[0].set_ylabel(f'Product similarity ({product_column})')
    fig.suptitle(f'Retention of pairwise similarity after attachment: {product_column}', y=1.04)
    fig.tight_layout()
    return fig, axes

def plot_partner_correlations(per_partner_metrics):
    """Show how global correlation varies across the ten partners."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for descriptor, group in per_partner_metrics.groupby('descriptor', sort=False):
        group = group.copy()
        group['partner_number'] = group['product_column'].str.extract(r'(\d+)').astype(int)
        group = group.sort_values('partner_number')
        for ax, metric, title in zip(axes, ['spearman', 'pearson'], ['Spearman rank correlation', 'Pearson correlation']):
            ax.plot(
                group['product_column'], group[metric], marker='o',
                label=DESCRIPTOR_LABELS[descriptor], color=DESCRIPTOR_COLORS[descriptor],
            )
            ax.set_title(title)
            ax.set_xlabel('Constant attachment partner')
            ax.tick_params(axis='x', rotation=45)
            ax.grid(axis='y', alpha=0.25)
    axes[0].set_ylabel('Correlation with building-block similarity')
    axes[1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    return fig, axes

def plot_final_summary(descriptor_summary):
    """Plot mean performance across partners with between-partner SD."""
    metrics = [
        ('spearman', 'Spearman ρ'),
        ('top_5_neighbor_preservation', 'Top-5 preservation'),
        ('top_10_neighbor_preservation', 'Top-10 preservation'),
    ]
    descriptors = descriptor_summary['descriptor'].tolist()
    x = np.arange(len(metrics))
    width = 0.19
    fig, ax = plt.subplots(figsize=(10, 4.8))

    for index, descriptor in enumerate(descriptors):
        row = descriptor_summary.loc[descriptor_summary['descriptor'] == descriptor].iloc[0]
        means = [row[f'{metric}_mean'] for metric, _ in metrics]
        errors = [row[f'{metric}_std'] for metric, _ in metrics]
        ax.bar(
            x + (index - 1.5) * width, means, width, yerr=errors, capsize=3,
            label=DESCRIPTOR_LABELS[descriptor], color=DESCRIPTOR_COLORS[descriptor],
        )

    ax.set_xticks(x, [label for _, label in metrics])
    ax.set_ylabel('Mean across ten attachment partners')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.25)
    ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    return fig, ax

### Pairwise correlation example

The four panels below show all unique molecular pairs for one attachment partner. A point on the diagonal has the same similarity before and after attachment. The panel title reports the rank correlation used in the quantitative comparison. Change `SMILES_0` to another product column to inspect a different partner.

In [ ]:
fig, axes = plot_correlation_panels(RESULT_DIR, product_column='SMILES_8')
save_figure(fig, 'pairwise_similarity_retention_SMILES_8.svg');

### Correlation across attachment partners

These plots show whether descriptor performance is consistent across the ten constant partners or driven by only one attachment geometry. Spearman is the primary correlation because common attached fragments can compress the similarity range and produce nonlinear relationships.

In [ ]:
fig, axes = plot_partner_correlations(results.per_partner_metrics)
save_figure(fig, 'correlations_across_attachment_partners.svg');

## Primary benchmark summary


In [ ]:
display(results.descriptor_summary.round(4))
fig, ax = plot_final_summary(results.descriptor_summary)
save_figure(fig, 'descriptor_performance_summary.svg');

## Differences relative to standard ECFP-O

The final table reports paired differences for the same attachment partner. Positive values mean that the named descriptor retained similarity or neighbors better than ECFP-O; negative values mean ECFP-O performed better. ECFP-Hg specifically tests whether retaining an explicit attachment marker is sufficient without directional propagation.

In [ ]:
display(results.paired_descriptor_differences.round(4))

## Sensitivity analysis: fingerprint radius

The primary benchmark is repeated in distinct run directories for each radius.
Existing result directories are never overwritten.


In [ ]:
RADIUS_DIR = PROJECT_ROOT / 'results' / 'step1_radius'
RADIUS_DIR.mkdir(parents=True, exist_ok=True)

def unused_sensitivity_directory(parent, name):
    output_dir = parent / name
    if output_dir.exists():
        raise FileExistsError(
            f'Results already exist: {output_dir}\n'
            'Choose another run name or inspect the existing results.'
        )
    return output_dir


In [ ]:
radius_results = {}

for radius in (2, 3, 4):
    config = BenchmarkConfig(
        n_bits=1024,
        ecfp_radius=radius,
        coaf_radius=radius,
        linear_max_path_length=2*radius,
        neighbor_k=(5, 10),
    )

    run_name = f"radius_{radius}_bits_1024"
    output_dir = unused_sensitivity_directory(RADIUS_DIR, run_name)

    radius_results[radius] = run_similarity_retention_benchmark(
        dataset,
        config=config,
        output_dir=output_dir,
    )

## Sensitivity analysis: fingerprint length

Bit-length runs are written below `results/step1_bitstring`, separately from
the radius analysis and the primary benchmark.


In [ ]:
BITSTRING_DIR = PROJECT_ROOT / "results" / "step1_bitstring"

def unused_bitstring_output_directory(name):
    output_dir = BITSTRING_DIR / name

    if output_dir.exists():
        raise FileExistsError(
            f"Results already exist: {output_dir}"
        )

    return output_dir

In [ ]:
bit_length_results = {}

for n_bits in (512, 1024, 2048, 4096):
    config = BenchmarkConfig(
        n_bits=n_bits,
        ecfp_radius=3,
        coaf_radius=3,
        linear_max_path_length=6,
        neighbor_k=(5, 10),
    )

    run_name = f"bits_{n_bits}"
    output_dir = unused_bitstring_output_directory(run_name)

    bit_length_results[n_bits] = run_similarity_retention_benchmark(
        dataset,
        config=config,
        output_dir=output_dir,
    )